# 08 – Modeling POINT_TOTAL (Regression)

Entrainer des modèles de régression (LightGBM, XGBoost, stacking) sur le dataset gold `POINT_TOTAL`, avec logging complet dans MLflow. Assure-toi d'avoir généré `gold_dataset_point_total_*.parquet` via `06_build_datasets.ipynb` ou le CLI avant d'exécuter ce notebook.

In [1]:
import pandas as pd
import numpy as np

from src.modeling import DatasetConfig, TrainingConfig, ModelTrainer, tune_point_total

dataset_cfg = DatasetConfig(
    target="POINT_TOTAL",
    gold_pattern="gold_dataset_point_total_*.parquet",
)

training_cfg = TrainingConfig(
    experiment_name="point_total_regression",
    tracking_uri="file:./mlruns",
    test_size=0.2,
    random_state=42,
    stratify=False,
    task_type="regression",
    pivot_value=220.0,
    enable_learning_curve=False,
    pivot_values=[218.5, 219.5, 220.5, 221.5, 222.5, 223.5, 224.5, 225.5, 226.5, 227.5, 228.5, 229.5, 230.5, 231.5, 232.5, 233.5, 234.5, 235.5, 236.5, 237.5, 238.5, 239.5, 240.5, 241.5, 242.5, 243.5, 244.5],
    enable_sigma_model=True,
    min_sigma=6.0,
)

trainer = ModelTrainer(dataset_cfg, training_cfg)
print("Dataset loaded from:", dataset_cfg.resolve_path())
print("Train shape:", trainer.X_train.shape, "- Test shape:", trainer.X_test.shape)
print("Target stats:", trainer.y_train.describe())

/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset loaded from: data/03_gold/gold_dataset_point_total_20251110T222845Z.parquet
Train shape: (25792, 280) - Test shape: (6449, 280)
Target stats: count    25792.000000
mean       205.275202
std         24.113739
min        121.000000
25%        188.000000
50%        204.000000
75%        221.000000
max        351.000000
Name: POINT_TOTAL, dtype: float64


## Tuning Optuna (optionnel)

Activez ce bloc pour lancer quelques essais Optuna avant l'entraînement final.


In [ ]:
from pathlib import Path
import json

ENABLE_TUNING = False  # Passez à True pour relancer Optuna
TUNING_MODELS = ["lgbm", "xgb", "ngboost", "stacking"]
OPTUNA_TRIALS = 20
BEST_PARAMS_PATH = Path("artifacts/point_total_best_params.json")

MODEL_REGISTRY = {
    "lgbm": {"default": True},
    "xgb": {"default": True},
    "ngboost": {"default": False},
    "stacking": {"default": True},
}

LEGACY_KEY_FIXES = {
    "stacking": {"stack_alpha": "final_estimator__alpha"},
}


def normalize_params(model_key: str, params: dict) -> dict:
    mapping = LEGACY_KEY_FIXES.get(model_key, {})
    normalized = {}
    for key, value in params.items():
        normalized[mapping.get(key, key)] = value
    return normalized


best_params = {}
if ENABLE_TUNING:
    tuning_results = tune_point_total(
        dataset_cfg,
        training_cfg,
        models=TUNING_MODELS,
        n_trials=OPTUNA_TRIALS,
    )
    for res in tuning_results:
        print(f"{res.model} -> {res.best_value:.4f}")
        best_params[res.model] = normalize_params(res.model, res.best_params)
    BEST_PARAMS_PATH.parent.mkdir(parents=True, exist_ok=True)
    BEST_PARAMS_PATH.write_text(json.dumps({k: {"params": v} for k, v in best_params.items()}, indent=2))
elif BEST_PARAMS_PATH.exists():
    cache = json.loads(BEST_PARAMS_PATH.read_text())
    best_params = {k: normalize_params(k, entry.get("params", entry)) for k, entry in cache.items()}
    print("Chargement des hyperparamètres Optuna depuis le cache.")
else:
    print("Pas d'hyperparamètres tunés : utilisation des valeurs par défaut.")


def get_models_to_train(best_params, registry):
    if best_params:
        return list(best_params.keys())
    return [name for name, meta in registry.items() if meta.get("default", False)]


Chargement des hyperparamètres Optuna depuis le cache.


## Entraînement & logging

Lance les modèles choisis (tuning ou défaut) et consigne les artefacts.


In [3]:
MODELS_TO_TRAIN = get_models_to_train(best_params, MODEL_REGISTRY)
print('Modèles entraînés :', MODELS_TO_TRAIN)

rows = []
for model_key in MODELS_TO_TRAIN:
    overrides = best_params.get(model_key)
    tuned = overrides is not None
    run_name = f"{model_key}_tuned" if tuned else model_key
    metrics = trainer.train_with_params(
        model_key,
        param_overrides=overrides,
        run_name=run_name,
    )
    rows.append({"model": model_key, "tuned": tuned, **metrics})

results_df = pd.DataFrame(rows)
display(results_df)


Modèles entraînés : ['lgbm', 'xgb', 'ngboost', 'stacking']
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.073315 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12914
[LightGBM] [Info] Number of data points in the train set: 25792, number of used features: 103
[LightGBM] [Info] Start training from score 205.275202


/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006672 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12914
[LightGBM] [Info] Number of data points in the train set: 25792, number of used features: 103
[LightGBM] [Info] Start training from score 16.847584


/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2025/11/11 20:10:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/11 20:10:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.068625 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12914
[LightGBM] [Info] Number of data points in the train set: 25792, number of used features: 103
[LightGBM] [Info] Start training from score 15.714874


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.043095 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12914
[LightGBM] [Info] Number of data points in the train set: 25792, number of used features: 103
[LightGBM] [Info] Start training from score 205.275202
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018684 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12792
[LightGBM] [Info] Number of data points in the train set: 20634, number of used features: 103
[LightGBM] [Info] Start training from score 205.340506
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017066 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12784
[LightGBM] [Info] Auto-choosing col-wise multi-

/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarnin

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004119 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12914
[LightGBM] [Info] Number of data points in the train set: 25792, number of used features: 103
[LightGBM] [Info] Start training from score 13.931054


/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2025/11/11 20:26:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/11 20:26:06 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


,model,tuned,rmse,mae,r2
0,lgbm,True,21.492992,17.043358,0.217829
1,xgb,True,21.504698,17.026537,0.216977
2,ngboost,True,21.619026,17.139879,0.208629
3,stacking,True,21.474067,16.995456,0.219206


Chaque run consigne RMSE/MAE/R², les graphiques prédiction vs réalité, l'histogramme des résidus et le modèle sérialisé dans MLflow (`mlruns`).